# PDF RAG System — Study Notebook

A minimal Retrieval-Augmented Generation pipeline over a PDF.

**Flow:** `PDF → text → chunks → embeddings → vector DB → retrieve → LLM → answer`

## Prerequisites
1. `uv sync` (installs from `pyproject.toml`)
2. In VSCode: select the kernel from `.venv/bin/python`
3. Set `ANTHROPIC_API_KEY` in a `.env` file (or env var) for the generation step
4. Drop a PDF at `./data/sample.pdf` (or change the path below)

## Step 1 — Imports & config

We load:
- `pypdf` to extract PDF text
- `langchain_text_splitters` for chunking
- `sentence_transformers` for local embeddings (no API needed)
- `chromadb` as a local vector store
- `anthropic` to call Claude for the answer

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from anthropic import Anthropic

load_dotenv()

PDF_PATH = Path("data/sample.pdf")
DB_DIR = Path("db")
COLLECTION = "pdf_chunks"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CLAUDE_MODEL = "claude-haiku-4-5"

DB_DIR.mkdir(exist_ok=True)
PDF_PATH.parent.mkdir(exist_ok=True)
print("OK")

## Step 2 — Extract text from PDF

We iterate page-by-page and keep the page number alongside each piece of text. Page numbers become useful later as **citations** in the answer.

In [ ]:
def load_pdf(path: Path) -> list[dict]:
    reader = PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"page": i, "text": text})
    return pages

pages = load_pdf(PDF_PATH)
print(f"Loaded {len(pages)} pages")
print("First 300 chars of page 1:\n", pages[0]["text"][:300])

## Step 3 — Chunk the text

Why chunk? LLMs and vector search work best on focused passages, not whole pages.

- `chunk_size=800` — characters per chunk
- `chunk_overlap=100` — keeps context across boundaries so sentences don't get split awkwardly

Tweak these values and observe how retrieval quality changes.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks: list[dict] = []
for p in pages:
    for piece in splitter.split_text(p["text"]):
        chunks.append({"page": p["page"], "text": piece})

print(f"{len(chunks)} chunks")
print("Sample chunk:\n", chunks[0])

## Step 4 — Embed chunks

An **embedding** is a vector capturing semantic meaning. Similar meaning → close in vector space.

We use `all-MiniLM-L6-v2` (384-dim, fast, runs on CPU). The model downloads on first use.

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL)
texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print("Shape:", embeddings.shape)

## Step 5 — Store in Chroma

Chroma is a local vector DB. We persist to `./db` so the index survives kernel restarts.

Each entry has: `id`, `embedding`, the chunk `text` (document), and `metadata` (page number).

In [ ]:
client = chromadb.PersistentClient(path=str(DB_DIR))

if COLLECTION in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION)
collection = client.create_collection(COLLECTION)

collection.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{"page": c["page"]} for c in chunks],
)
print("Indexed:", collection.count())

## Step 6 — Retrieve relevant chunks

Given a question:
1. Embed it with the **same** model used in Step 4 (critical!)
2. Ask Chroma for the top-k nearest chunks
3. Return their text + page metadata

In [ ]:
def retrieve(question: str, k: int = 4) -> list[dict]:
    q_vec = embedder.encode([question], convert_to_numpy=True).tolist()
    res = collection.query(query_embeddings=q_vec, n_results=k)
    return [
        {"text": doc, "page": meta["page"], "distance": dist}
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

hits = retrieve("What is the main topic of this document?")
for h in hits:
    print(f"[page {h['page']}] dist={h['distance']:.3f}  {h['text'][:120]}...")

## Step 7 — Generate an answer with Claude

We build a prompt that:
- Pins the model to **only** use retrieved context (reduces hallucinations)
- Includes page numbers so the model can cite sources
- Tells it to say `I don't know` if the answer isn't in the context

In [ ]:
anthropic_client = Anthropic()

PROMPT = """You are answering questions about a PDF using ONLY the context below.
If the answer is not in the context, say: "I don't know based on the provided document."
Cite sources inline using [page N].

Context:
{context}

Question: {question}
"""

def answer(question: str, k: int = 4) -> str:
    hits = retrieve(question, k=k)
    context = "\n\n".join(f"[page {h['page']}]\n{h['text']}" for h in hits)
    msg = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=800,
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
    )
    return msg.content[0].text

print(answer("Summarize this document in 3 bullet points."))

## Step 8 — Try it

Ask your own questions. Things to experiment with:
- Change `k` (more chunks = more context but noisier)
- Change `chunk_size` / `chunk_overlap` in Step 3 and re-run from there
- Swap `EMBED_MODEL` to a larger one (e.g. `BAAI/bge-base-en-v1.5`)
- Compare answers when you ask something **not** in the PDF — does it correctly say "I don't know"?

In [ ]:
print(answer("What are the key concepts mentioned?"))

## Where to go next

1. **Hybrid search** — combine vector + BM25 keyword search for better recall on names/IDs
2. **Re-ranking** — use a cross-encoder (e.g. `bge-reranker-base`) to reorder top-k before sending to the LLM
3. **Query rewriting** — have Claude rephrase the question before retrieval
4. **Multi-document** — add a `source` filename to metadata and ingest many PDFs
5. **Evaluation** — measure retrieval recall@k and faithfulness with RAGAS